In [1]:
import pandas as pd
from job import make_predictions
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone
import os

load_dotenv() 


True

In [2]:
df = pd.read_csv("training_data.csv")

In [3]:
def convert_filename_to_datetime(filename):
    try:
        # Extract date and time from the filename
        filename = filename.split('/')[-1]
        date_str = filename[5:15]  # Extract 'YYYY.MM.DD'
        time_str = filename[16:23]  # Extract 'HH.MM.SS'

        # Create a datetime object
        datetime_str = f"{date_str} {time_str}"
        formatted_datetime = datetime.strptime(datetime_str, '%Y.%m.%d %H.%M.%S')

        # Convert to desired format
        result = formatted_datetime.strftime('%Y-%m-%d %H:%M:%S')
        return datetime.strptime(result,os.getenv("date_format")) 

    except Exception as e:
        print(f"Error converting filename to datetime: {e}")
        return None

In [4]:
# Apply the function to the 'filename' column
df['datetime'] = df['timestamp'].apply(convert_filename_to_datetime)
df['datetime'][0]


Timestamp('2011-02-15 00:00:00')

In [5]:
df = df[df.flare_prob >= 0.5]

In [6]:
# df['predictions'] = df['datetime'].apply(make_predictions)

In [10]:
import numpy as np
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

def normalize_and_scale_array(array, max_pixel=255):
    """
    Normalize and scale a given array to the range [0, max_pixel].

    Args:
        array (numpy.ndarray): The input array to be normalized and scaled.
        max_pixel (int): The maximum pixel value for scaling. Default is 255.

    Returns:
        numpy.ndarray: The normalized and scaled array.
    """
    min_val = array.min()
    max_val = array.max()
    normalized_array = (array - min_val) / (max_val - min_val)
    scaled_array = (normalized_array * max_pixel).astype(np.uint8)
    return scaled_array

def denoise(array, lower_threshold, upper_threshold, max_pixel):
    """
    Denoise a given array by setting values below the lower threshold to 0
    and values above the upper threshold to max_pixel.

    Args:
        array (numpy.ndarray): The input array to be denoised.
        lower_threshold (float): Values below this threshold will be set to 0.
        upper_threshold (float): Values above this threshold will be set to max_pixel.
        max_pixel (int): The maximum pixel value.

    Returns:
        numpy.ndarray: The denoised array.
    """
    denoised_array = np.where(array < lower_threshold, 0,
                              np.where(array > upper_threshold, max_pixel, array))
    return denoised_array


def pearson_correlation(array1, array2):
    # Flatten the arrays for Euclidean distance calculation
    array1 = array1.flatten()
    array2 = array2.flatten()

    # Calculate Pearson correlation coefficient
    pearson_corr, _ = pearsonr(array1, array2)
    return pearson_corr

def inverse_euclidean(array1,array2):
    # Flatten the arrays for Euclidean distance calculation
    array1 = array1.flatten()
    array2 = array2.flatten()

    # Calculate Euclidean distance
    euclidean_dist = euclidean(array1, array2)

    # Calculate inverse of Euclidean distance as similarity (bounded between 0 and 1)
    euclidean_sim = 1 / (1 + euclidean_dist)
    return euclidean_sim

def compare_arrays(row):
    """
    Compare two arrays using Pearson correlation coefficient and inverse Euclidean distance.

    Args:
        array1 (numpy.ndarray): First input array.
        array2 (numpy.ndarray): Second input array.

    Returns:
        dict: A dictionary containing the results of comparisons.
    """
    try:
        results = {}

        array1 = np.load(row['artefacts']['intgrad'])
        array2 = np.load(row['artefacts']['guidedgradcam'])

        # Scaled Comparison
        scaled_array1 = normalize_and_scale_array(array1)
        scaled_array2 = normalize_and_scale_array(array2)
        scaled_pearson_corr = pearson_correlation(scaled_array1, scaled_array2)
        scaled_inv_euclidean = inverse_euclidean(scaled_array1, scaled_array2)
        results['scaled_pearson_corr'] = scaled_pearson_corr
        results['scaled_inv_euclidean'] = scaled_inv_euclidean

        # Denoised Comparison (Thresholds: 30, 70, 255)
        denoised_array1 = denoise(scaled_array1, 30, 70, 255)
        denoised_array2 = denoise(scaled_array2, 30, 70, 255)
        denoised_pearson_corr = pearson_correlation(denoised_array1, denoised_array2)
        denoised_inv_euclidean = inverse_euclidean(denoised_array1, denoised_array2)
        results['denoised_pearson_corr'] = denoised_pearson_corr
        results['denoised_inv_euclidean'] = denoised_inv_euclidean

        # Denoised Comparison (Thresholds: 30, 30, 255)
        denoised_array1_2 = denoise(scaled_array1, 30, 30, 255)
        denoised_array2_2 = denoise(scaled_array2, 30, 30, 255)
        denoised_pearson_corr_2 = pearson_correlation(denoised_array1_2, denoised_array2_2)
        denoised_inv_euclidean_2 = inverse_euclidean(denoised_array1_2, denoised_array2_2)
        results['denoised_pearson_corr_2'] = denoised_pearson_corr_2
        results['denoised_inv_euclidean_2'] = denoised_inv_euclidean_2

        return results
    except Exception as e:
        return None


In [11]:
df['comparison'] = df['predictions'].apply(compare_arrays)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 794 entries, 0 to 879
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   timestamp     794 non-null    object        
 1   goes_class_x  794 non-null    object        
 2   flare_prob    794 non-null    float64       
 3   flare_start   794 non-null    object        
 4   fl_lon        794 non-null    float64       
 5   fl_lat        794 non-null    float64       
 6   rest_fl       794 non-null    object        
 7   rest_lon      794 non-null    object        
 8   rest_lat      794 non-null    object        
 9   datetime      794 non-null    datetime64[ns]
 10  predictions   794 non-null    object        
 11  comparison    488 non-null    object        
dtypes: datetime64[ns](1), float64(3), object(8)
memory usage: 112.9+ KB


In [28]:
import json

df1 = df[~df.comparison.isna()]
# Custom JSON Encoder
class CustomEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.float32):
            return float(obj)
        elif isinstance(obj, pd.Timestamp):
            return obj.strftime('%Y-%m-%d %H:%M:%S')
        return json.JSONEncoder.default(self, obj)

# Convert DataFrame to list of dictionaries
list_of_dicts = df1.to_dict(orient='records')

# Specify the file path
json_file = 'comparison.json'

# Save the list of dictionaries to a JSON file with custom encoder
with open(json_file, 'w') as f:
    json.dump(list_of_dicts, f, cls=CustomEncoder)


In [29]:
# from job_copy import make_predictions as mp
# art = mp(df['datetime'][794])